# Math Prerequisites → Probability | Chapter 2: MLE vs MAP Estimation

> **Why this chapter exists:** MLE and MAP are the two workhorses of parameter estimation in ML. Virtually every model you have ever trained uses one of them. Understanding them from first principles — and understanding that MAP is just MLE with a regularizer — is foundational for understanding Bayesian methods, regularization, and the bias-variance tradeoff. This chapter derives both from scratch.

---

## 1. The Parameter Estimation Problem

Given a dataset $\mathcal{D} = \{x_1, x_2, ..., x_N\}$ assumed drawn i.i.d. from some distribution $p(x | \theta)$, the task is to **infer** the unknown parameter(s) $\theta$.

There are three major approaches:

| Approach | Formula | Output | Philosophy |
| :--- | :--- | :--- | :--- |
| **MLE** | $\hat\theta = \arg\max_\theta p(\mathcal{D}|\theta)$ | Point estimate | Frequentist: parameters are fixed unknowns |
| **MAP** | $\hat\theta = \arg\max_\theta p(\theta|\mathcal{D})$ | Point estimate | Bayesian: parameters have a prior distribution |
| **Full Bayesian** | $p(\theta|\mathcal{D}) = \frac{p(\mathcal{D}|\theta)p(\theta)}{p(\mathcal{D})}$ | Full distribution | Bayesian: track uncertainty in $\theta$ |

This chapter covers MLE and MAP. Full Bayesian inference is covered in the `bayes/` series.

---

## 2. Maximum Likelihood Estimation (MLE)

### **The Idea**
Find the parameter $\theta$ that makes the observed data **most probable**:
$$\hat\theta_{\text{MLE}} = \arg\max_\theta p(\mathcal{D}|\theta) = \arg\max_\theta \prod_{i=1}^N p(x_i|\theta)$$

### **The Log-Likelihood Trick**
Maximizing a product is numerically unstable (product of many small numbers → underflow). Take the log, which is monotone increasing (so same argmax):
$$\hat\theta_{\text{MLE}} = \arg\max_\theta \sum_{i=1}^N \log p(x_i|\theta) = \arg\max_\theta \mathcal{L}(\theta)$$

The **log-likelihood** $\mathcal{L}(\theta) = \sum_i \log p(x_i|\theta)$.

---

### **2.1 MLE for Gaussian — Full Derivation**
Given $x_1, ..., x_N \sim \mathcal{N}(\mu, \sigma^2)$, find $\hat\mu$ and $\hat\sigma^2$.

**Log-likelihood:**
$$\mathcal{L}(\mu, \sigma^2) = -\frac{N}{2}\log(2\pi) - \frac{N}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum_{i=1}^N (x_i - \mu)^2$$

**Step 1: Optimize over $\mu$** (take $\partial\mathcal{L}/\partial\mu = 0$):
$$\frac{\partial\mathcal{L}}{\partial\mu} = \frac{1}{\sigma^2}\sum_{i=1}^N (x_i - \mu) = 0 \implies \boxed{\hat\mu_{\text{MLE}} = \frac{1}{N}\sum_{i=1}^N x_i = \bar{x}}$$

The MLE for the mean is the sample mean. Makes sense.

**Step 2: Optimize over $\sigma^2$** (let $v = \sigma^2$, take $\partial\mathcal{L}/\partial v = 0$):
$$\frac{\partial\mathcal{L}}{\partial v} = -\frac{N}{2v} + \frac{1}{2v^2}\sum_{i=1}^N(x_i-\mu)^2 = 0 \implies \boxed{\hat\sigma^2_{\text{MLE}} = \frac{1}{N}\sum_{i=1}^N (x_i - \bar{x})^2}$$

> ⚠️ **Biased!** The MLE variance uses $\frac{1}{N}$, not $\frac{1}{N-1}$. It systematically **underestimates** the true population variance. That's why sample variance uses Bessel's correction ($N-1$).

---

### **2.2 MLE for Bernoulli — Full Derivation**
Given $x_1, ..., x_N \sim \text{Bernoulli}(\theta)$, find $\hat\theta$.

**Log-likelihood:**
$$\mathcal{L}(\theta) = \sum_i \left[x_i \log\theta + (1-x_i)\log(1-\theta)\right] = k\log\theta + (N-k)\log(1-\theta)$$

where $k = \sum_i x_i$ (number of successes).

**Optimize:** $\frac{d\mathcal{L}}{d\theta} = \frac{k}{\theta} - \frac{N-k}{1-\theta} = 0$

Solving: $k(1-\theta) = (N-k)\theta \Rightarrow k = N\theta \Rightarrow \boxed{\hat\theta_{\text{MLE}} = \frac{k}{N}}$

The MLE for a coin's bias = observed fraction of heads. Intuitive!

---

### **2.3 MLE Connections to Loss Functions**

| Noise Model | Negative Log-Likelihood | Equivalent Loss Function |
| :--- | :--- | :--- |
| Gaussian: $y \sim \mathcal{N}(f(x), \sigma^2)$ | $\sum_i (y_i - f(x_i))^2 / (2\sigma^2)$ | **Mean Squared Error (MSE)** |
| Bernoulli: $y \sim \text{Bernoulli}(\sigma(f(x)))$ | $-\sum_i [y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$ | **Binary Cross-Entropy** |
| Categorical: $y \sim \text{Cat}(\text{softmax}(f(x)))$ | $-\sum_i \sum_k y_{ik}\log\hat{p}_{ik}$ | **Categorical Cross-Entropy** |

> **Key insight:** When you minimize MSE, you are implicitly assuming Gaussian noise and performing MLE. When you minimize cross-entropy, you are assuming Bernoulli/Categorical likelihood and performing MLE. **All standard training losses are MLE in disguise.**

---

## 3. Maximum A Posteriori (MAP) Estimation

### **The Idea**
Instead of maximizing just the likelihood, maximize the **posterior** — the product of likelihood and prior:
$$\hat\theta_{\text{MAP}} = \arg\max_\theta p(\theta|\mathcal{D}) = \arg\max_\theta p(\mathcal{D}|\theta) \cdot p(\theta)$$

Taking the log:
$$\hat\theta_{\text{MAP}} = \arg\max_\theta \left[\underbrace{\sum_i \log p(x_i|\theta)}_{\text{log-likelihood}} + \underbrace{\log p(\theta)}_{\text{log-prior}}\right]$$

**MAP = MLE + log-prior regularizer.** The prior acts as a penalty that pulls the estimate toward what we believe $\theta$ should be.

---

### **3.1 MAP with Gaussian Prior → L2 Regularization (Ridge)**

Consider linear regression with weights $\vec{w}$ and Gaussian prior $\vec{w} \sim \mathcal{N}(0, \tau^2 I)$:

$$\hat{\vec{w}}_{\text{MAP}} = \arg\max_{\vec{w}} \underbrace{-\frac{1}{2\sigma^2}\|\vec{y} - X\vec{w}\|^2}_{\text{Gaussian likelihood}} \underbrace{- \frac{1}{2\tau^2}\|\vec{w}\|^2}_{\text{Gaussian prior}}$$

Multiplying through by $-2\sigma^2$:
$$\hat{\vec{w}}_{\text{MAP}} = \arg\min_{\vec{w}} \|\vec{y} - X\vec{w}\|^2 + \underbrace{\frac{\sigma^2}{\tau^2}}_{\lambda} \|\vec{w}\|^2$$

$$\boxed{\hat{\vec{w}}_{\text{MAP}} = \arg\min_{\vec{w}} \text{MSE}(\vec{w}) + \lambda \|\vec{w}\|_2^2 \quad \text{(Ridge Regression / L2 Regularization)}}$$

The regularization strength $\lambda = \sigma^2/\tau^2$:
- Large $\tau^2$ (weak/diffuse prior) → small $\lambda$ → MAP ≈ MLE
- Small $\tau^2$ (strong/tight prior near 0) → large $\lambda$ → heavy regularization

**Ridge regression IS MAP estimation with a Gaussian prior on weights.**

---

### **3.2 MAP with Laplace Prior → L1 Regularization (Lasso)**

The Laplace distribution: $p(w) = \frac{1}{2b}\exp\left(-\frac{|w|}{b}\right)$

With Laplace prior on weights:
$$\hat{\vec{w}}_{\text{MAP}} = \arg\min_{\vec{w}} \|\vec{y} - X\vec{w}\|^2 + \underbrace{\frac{2\sigma^2}{b}}_{\lambda} \|\vec{w}\|_1$$

$$\boxed{\hat{\vec{w}}_{\text{MAP}} = \arg\min_{\vec{w}} \text{MSE}(\vec{w}) + \lambda\|\vec{w}\|_1 \quad \text{(Lasso / L1 Regularization)}}$$

L1 induces **sparsity** (many weights exactly zero) because the Laplace distribution has a sharp peak at 0 — it places more probability density near 0 than the Gaussian does.

**Lasso IS MAP estimation with a Laplace prior on weights.**

---

## 4. MLE vs MAP vs Full Bayesian

| Aspect | MLE | MAP | Full Bayesian |
| :--- | :--- | :--- | :--- |
| Output | Point $\hat\theta$ | Point $\hat\theta$ | Distribution $p(\theta|\mathcal{D})$ |
| Uses prior? | ❌ | ✅ | ✅ |
| Uncertainty in $\theta$? | ❌ | ❌ | ✅ |
| Small $N$ behavior | Overfits | Regularized by prior | Posterior stays wide |
| Large $N$ behavior | Consistent | Consistent (prior dominated by data) | Concentrated posterior \approx MLE |
| Computation | Usually analytical or convex | Usually analytical or convex | Often intractable (needs MCMC/VI) |
| Regularization link | No regularization | ✅ MAP ↔ regularized MLE | ✅ Prior = implicit regularizer |

### **The Bayesian Lens on Bias-Variance**
- **MLE** has low bias (uses all data flexibility) but high variance (overfits on small $N$).
- **MAP** with strong prior has higher bias (pulled toward prior) but lower variance (less overfitting).
- As $N \to \infty$: MLE and MAP converge (data swamps the prior).
- As $N \to 0$: MAP approaches the prior mean; MLE is undefined or extreme.

---

## 5. Interview Q&A

**Q1: What is MLE in one sentence?**
> MLE finds the parameter $\theta$ that maximizes the probability of observing the given data: $\hat\theta = \arg\max_\theta \prod_i p(x_i|\theta)$.

**Q2: Why use log-likelihood instead of likelihood?**
> Taking the log converts the product into a sum ($\log\prod = \sum\log$), which: (1) avoids numerical underflow from multiplying many small probabilities; (2) is mathematically easier to differentiate; (3) doesn't change the argmax since log is monotone increasing.

**Q3: What is MAP and how does it differ from MLE?**
> MAP maximizes the posterior $p(\theta|\mathcal{D}) \propto p(\mathcal{D}|\theta) p(\theta)$ instead of just the likelihood. The difference is the log-prior term $\log p(\theta)$, which acts as a regularizer pulling the estimate toward the prior. MAP = MLE with regularization.

**Q4: What prior corresponds to L2 regularization (Ridge)?**
> A Gaussian prior $\vec{w} \sim \mathcal{N}(0, \tau^2 I)$. The log-prior is $-\|\vec{w}\|^2/(2\tau^2)$, which after negation becomes the $\lambda\|\vec{w}\|^2$ Ridge penalty with $\lambda = \sigma^2/\tau^2$.

**Q5: What prior corresponds to L1 regularization (Lasso)?**
> A Laplace prior $p(w_j) \propto \exp(-|w_j|/b)$. The log-prior is $-\sum_j |w_j|/b$, which becomes the $\lambda\|\vec{w}\|_1$ Lasso penalty. The Laplace prior has heavier tails and a sharper peak at 0 than Gaussian, encouraging exact sparsity.

**Q6: Why does MLE overfit on small datasets?**
> MLE maximizes the likelihood of the specific observed data. With small $N$, the training data may not represent the population well, so MLE finds parameters that fit the noise in the training set. A prior (MAP) constrains the estimate, effectively reducing the model's flexibility and lowering variance.

**Q7: When does MAP = MLE?**
> When the prior is flat (uniform) over all parameter values: $p(\theta) = \text{const}$. Then $\log p(\theta)$ adds a constant to the log-posterior and doesn't affect the argmax. Also, MAP → MLE as $N \to \infty$ (data dominates prior for any fixed prior).

**Q8: Is minimizing cross-entropy equivalent to MLE?**
> Yes. Binary cross-entropy = negative log-likelihood under a Bernoulli model. Categorical cross-entropy = negative log-likelihood under a Categorical (softmax) model. Training a neural network with cross-entropy loss = performing MLE under the corresponding noise model.

---

In [ ]:
# ─── CELL 1: MLE vs MAP — Gaussian Example with Small N ──────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
true_mu = 3.0
true_sigma = 2.0
N_values = [3, 10, 30, 100, 500]

# Prior: mu ~ N(0, tau^2)
prior_mu   = 0.0   # prior mean
prior_tau  = 1.0   # prior std — strong prior

mle_estimates = []
map_estimates = []

for N in N_values:
    samples = np.random.normal(true_mu, true_sigma, N)
    # MLE: sample mean
    mu_mle = samples.mean()
    # MAP: precision-weighted avg of prior mean and sample mean
    # Posterior mean: (prior precision × prior mean + data precision × sample mean) / (sum of precisions)
    prior_prec = 1 / prior_tau**2
    data_prec  = N / true_sigma**2
    mu_map = (prior_prec * prior_mu + data_prec * mu_mle) / (prior_prec + data_prec)
    mle_estimates.append(mu_mle)
    map_estimates.append(mu_map)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Left: MLE vs MAP estimates as N grows
axes[0].plot(N_values, mle_estimates, 'o-', color='steelblue', lw=2, ms=8, label='MLE')
axes[0].plot(N_values, map_estimates, 's-', color='darkorange', lw=2, ms=8, label='MAP')
axes[0].axhline(true_mu, color='red', ls='--', lw=1.5, label=f'True μ={true_mu}')
axes[0].axhline(prior_mu, color='gray', ls=':', alpha=0.7, label=f'Prior mean={prior_mu}')
axes[0].set_xscale('log'); axes[0].set_xlabel('N (log scale)', fontsize=11)
axes[0].set_ylabel('Estimated μ', fontsize=11)
axes[0].set_title('MLE vs MAP: Gaussian Mean Estimation\nMAP is pulled toward prior, MLE is not', fontsize=10)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Middle: L2 vs L1 regularization as priors
x = np.linspace(-4, 4, 300)
gauss_prior = stats.norm.pdf(x, 0, 1)  # Gaussian prior → Ridge
laplace_prior = stats.laplace.pdf(x, 0, 1)  # Laplace prior → Lasso
axes[1].plot(x, gauss_prior, color='steelblue', lw=2.5, label='Gaussian prior → L2/Ridge')
axes[1].plot(x, laplace_prior, color='darkorange', lw=2.5, label='Laplace prior → L1/Lasso')
axes[1].fill_between(x, laplace_prior, alpha=0.15, color='darkorange')
axes[1].fill_between(x, gauss_prior, alpha=0.15, color='steelblue')
axes[1].set_title('Priors and Their Regularization Correspondence\n'
                  'Laplace: sharper at 0 → encourages sparsity', fontsize=10)
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Parameter value w', fontsize=11)
axes[1].set_ylabel('Prior density p(w)', fontsize=11)

# Right: MLE bias demonstration — sampling distribution
np.random.seed(0)
n_experiments = 1000
N_small = 5
mle_samples = np.array([np.random.normal(true_mu, true_sigma, N_small).mean()
                         for _ in range(n_experiments)])
map_samples = np.array([]
    )
for _ in range(n_experiments):
    s = np.random.normal(true_mu, true_sigma, N_small)
    mu_mle_ = s.mean()
    data_prec_ = N_small / true_sigma**2
    mu_map_ = (prior_prec * prior_mu + data_prec_ * mu_mle_) / (prior_prec + data_prec_)
    map_samples = np.append(map_samples, mu_map_)

axes[2].hist(mle_samples, bins=40, alpha=0.6, color='steelblue', density=True, label=f'MLE (N={N_small})')
axes[2].hist(map_samples, bins=40, alpha=0.6, color='darkorange', density=True, label=f'MAP (N={N_small})')
axes[2].axvline(true_mu, color='red', lw=2, ls='--', label=f'True μ={true_mu}')
axes[2].axvline(mle_samples.mean(), color='steelblue', lw=1.5, ls=':', label=f'Avg MLE={mle_samples.mean():.2f}')
axes[2].axvline(map_samples.mean(), color='darkorange', lw=1.5, ls=':', label=f'Avg MAP={map_samples.mean():.2f}')
axes[2].set_title(f'Sampling Distribution ({n_experiments} experiments, N={N_small})\n'
                  'MAP is biased but lower variance', fontsize=10)
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.suptitle('MLE vs MAP: Bias-Variance Tradeoff through Bayesian Lens', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: MAP as Regularization — Ridge and Lasso Connections ───────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
N, d = 50, 30  # small N, many features → likely to overfit

# Only first 5 features are truly informative
true_w = np.zeros(d)
true_w[:5] = np.array([3., -2., 1.5, -1., 0.5])

X = np.random.randn(N, d)
y = X @ true_w + np.random.randn(N) * 0.5

# Scale features
scaler = StandardScaler().fit(X)
X_std = scaler.transform(X)

# Fit models
lr = LinearRegression().fit(X_std, y)   # MLE (no prior)
ridge = Ridge(alpha=5.0).fit(X_std, y)  # MAP with Gaussian prior
lasso = Lasso(alpha=0.3).fit(X_std, y)  # MAP with Laplace prior

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

x_vals = np.arange(d)
width = 0.25
for ax, title, models in [
    (axes[0], 'Sparse view (first 10 coefficients)', [(true_w[:10], 'True w', 'black'),
                                                       (lr.coef_[:10], 'MLE (OLS)', 'steelblue'),
                                                       (ridge.coef_[:10], 'MAP/Ridge', 'darkorange'),
                                                       (lasso.coef_[:10], 'MAP/Lasso', 'forestgreen')]),
]:
    for i, (coef, label, color) in enumerate(models):
        ax.bar(x_vals[:10] + (i-1.5)*width, coef, width, label=label, color=color, alpha=0.75)
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')
    ax.set_xlabel('Feature index'); ax.set_ylabel('Coefficient value')

# All coefficients: scatter plot
axes[1].scatter(true_w, lr.coef_,    color='steelblue',  alpha=0.7, s=40, label='MLE (OLS)')
axes[1].scatter(true_w, ridge.coef_, color='darkorange', alpha=0.7, s=40, label='MAP/Ridge')
axes[1].scatter(true_w, lasso.coef_, color='forestgreen',alpha=0.7, s=40, label='MAP/Lasso')
axes[1].plot([-3.5,4], [-3.5,4], 'k--', lw=1, alpha=0.5, label='Perfect estimate')
axes[1].set_xlabel('True coefficient'); axes[1].set_ylabel('Estimated coefficient')
axes[1].set_title('True vs Estimated Coefficients (all d features)\n'
                  'Lasso correctly zeroes noise features', fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Regularization path — Ridge
lambdas = np.logspace(-2, 3, 100)
ridge_coefs = np.array([Ridge(alpha=lam).fit(X_std, y).coef_ for lam in lambdas])
lasso_coefs = np.array([Lasso(alpha=lam, max_iter=5000).fit(X_std, y).coef_ for lam in lambdas])

axes[2].semilogx(lambdas, ridge_coefs[:, :5], '-', lw=1.5, alpha=0.8)  # first 5 (true) features
axes[2].semilogx(lambdas, ridge_coefs[:, 5:], '--', lw=0.8, alpha=0.3, color='gray')
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_xlabel('λ (regularization strength = σ²/τ²)', fontsize=10)
axes[2].set_ylabel('Coefficient value', fontsize=10)
axes[2].set_title('Ridge Regularization Path\nSolid=true features, Dashed=noise features', fontsize=10)
axes[2].grid(alpha=0.3)

plt.suptitle('MAP Estimation = Regularized MLE\nPrior type determines regularization type',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

print("True non-zero coefficients (first 5):")
print(f"  True:  {true_w[:5]}")
print(f"  OLS:   {lr.coef_[:5].round(3)}")
print(f"  Ridge: {ridge.coef_[:5].round(3)}")
print(f"  Lasso: {lasso.coef_[:5].round(3)}")
print(f"\nLasso non-zeros: {(lasso.coef_ != 0).sum()}/30  (true: 5 non-zero features)")